In [15]:
import csv
import re

In [16]:
records  = []
patterns = []

In [17]:
PIPELINES_FROM_REDMINES  = "save-issues\\redmine_issues.csv"
PIPELINES_PROCESSED      = "pipelinesWithApplicationCode.csv"

In [18]:
with open(PIPELINES_FROM_REDMINES, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for record in reader:
        develop_pipeline  = record.get("custom_URL_Job_Jenkins_Develop", "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        master_pipeline   = record.get("custom_URL_Job_Jenkins_Master",  "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        release_pipeline  = record.get("custom_URL_Job_Jenkins_Release", "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        it_pipeline       = record.get("custom_URL_Job_Jenkins_IT",      "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        lab_pipeline      = record.get("custom_URL_Job_Jenkins_Lab",     "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        main_pipeline     = record.get("custom_URL_Job_Jenkins_Main",    "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")

        match = re.search(r"[A-Z]\d+", record.get("custom_Codice_Applicativo", ""))
        code  = match.group(0) if match else ""

        pipelines = [ develop_pipeline, master_pipeline, release_pipeline, it_pipeline, lab_pipeline, main_pipeline ]
        for p in pipelines:
            if not p:
                continue
            parts = p.split("/", 1)
            if len(parts) != 2:
                continue

            record                    = {}
            record["gitlabGroup"]     = p.split("/")[0].strip()
            record["gitlabPproject"]  = p.split("/")[1].strip()
            record["jenkinsJobName"]  = p
            record["applicationCode"] = code
            records.append(record)

In [19]:
with open(PIPELINES_PROCESSED, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["gitlabGroup", "gitlabPproject", "jenkinsJobName", "applicationCode"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)